# GENIE BPC v1 BRCA - Data Collection, Recoding & Derived Variables

**Purpose:** Document the pipeline that produced `data/processed/extracted_variables_genie_data.csv` from the raw GENIE BPC v1 BRCA release.

**This notebook does not write data.** It records what was done, in order, with the exact recoding logic.

**Source release:** `/Users/robertjames/loc/data private/genie_bpc_datav1/` (38 files, ~25 MB total)

**Outputs documented here:**
- 30 per-component CSVs in `data/processed/genie_bpc_v1_*.csv`
- Two master joins: `genie_bpc_v1_clinical_master.csv` (cancer-level) and `genie_bpc_v1_sample_master.csv` (sample-level)
- `genie_bpc_v1_sample_master_full.csv` - sample_master + gnomeR gene_binary + pathways + non-index summary
- **Final analytic frame:** `extracted_variables_genie_data.csv` (1,234 samples x 2,108 cols)

**Build scripts (referenced, not re-executed here):**
- `src/data collection and processing/extract_genie_bpc_full.py`
- `src/data collection and processing/extract_genie_bpc_timelines.py`
- `src/data collection and processing/build_genie_bpc_sample_master_full.R`
- `src/data collection and processing/extract_variables_genie_data.py`

In [1]:
from pathlib import Path
import pandas as pd

PROJ = Path('/Users/robertjames/Documents/Documents - Robert\u2019s iMac/Research Projects/MSKCC Research Fellowship/Projects/genie_tcga_impact_brain_mets')
SRC = Path('/Users/robertjames/loc/data private/genie_bpc_datav1')
PROC = PROJ / 'data/processed'

## 1. Source inventory

GENIE BPC v1 BRCA was downloaded as 38 files into `loc/data private/genie_bpc_datav1/`. The release includes cBioPortal-style genomic files (`data_*.txt`), GENIE BPC analytic CSVs (`cancer_level_dataset_*.csv`, `patient_level_dataset.csv`, etc.), gene panel definitions (`data_gene_panel_*.txt`), and longitudinal timeline files (`data_timeline_*.txt`).

cBioPortal `.txt` clinical files prepend four `#`-commented header rows (DISPLAY_NAMES, DESCRIPTIONS, TYPES, PRIORITIES) before the actual column header. **All reads use `pd.read_csv(..., comment='#')` to skip these.**

In [2]:
# Inventory of source files used
files_used = [
    ('data_clinical_patient.txt',                    '1,134 rows x 42 cols',  'patient demographics + receptor status + workload'),
    ('data_clinical_supp_survival.txt',              '1,135 rows x 7 cols',   'OS + PFS_I_ADV + PFS_M_ADV'),
    ('data_clinical_supp_survival_treatment.txt',    '1,015 rows x 121 cols', 'per-regimen survival (skipped for brain-met analysis)'),
    ('patient_level_dataset.csv',                    '1,130 rows x 50 cols',  'GENIE BPC patient frame'),
    ('cancer_level_dataset_index.csv',               '1,141 rows x 185 cols', 'GOLDMINE: per-organ time-to-met + stage_dx_iv'),
    ('cancer_level_dataset_non_index.csv',           '191 rows x 112 cols',   'prior/concurrent non-breast cancers'),
    ('data_clinical_sample.txt',                     '1,227 rows x 9 cols',   'sample metadata'),
    ('cancer_panel_test_level_dataset.csv',          '1,234 rows x 29 cols',  'one row per panel test, links sample to cancer'),
    ('data_gene_matrix.txt',                         '1,223 rows x 3 cols',   'which panel applied per sample'),
    ('data_mutations_extended.txt',                  '7,051 rows x 64 cols',  'MAF with gnomAD AF, Polyphen, SIFT'),
    ('data_CNA.txt',                                 '1,003 genes x 1,199 samples (wide)', 'gene-level discrete CNA matrix'),
    ('data_cna_hg19.seg.txt',                        '42,760 rows x 6 cols',  'segment-level CNA'),
    ('data_sv.txt',                                  '427 rows x 40 cols',    'structural variants'),
    ('genomic_information.txt',                      '64,511 rows x 9 cols',  'per-panel covered regions'),
    ('regimen_cancer_level_dataset.csv',             '6,905 rows x 102 cols', 'treatment regimens per cancer'),
    ('data_timeline_cancer_diagnosis.txt',           '1,332 rows x 20 cols',  'longitudinal'),
    ('data_timeline_pathology.txt',                  '7,224 rows x 13 cols',  'longitudinal'),
    ('data_timeline_sample_acquisition.txt',         '1,233 rows x 8 cols',   'longitudinal'),
    ('data_timeline_sequencing.txt',                 '1,254 rows x 7 cols',   'longitudinal'),
    ('data_timeline_medonc.txt',                     '28,338 rows x 8 cols',  'longitudinal'),
    ('data_timeline_imaging.txt',                    '26,773 rows x 10 cols', 'longitudinal'),
    ('data_timeline_treatment.txt',                  '10,439 rows x 13 cols', 'longitudinal'),
    ('data_timeline_labtest.txt',                    '9,743 rows x 9 cols',   'longitudinal'),
    ('imaging_level_dataset.csv',                    '26,773 rows x 42 cols', 'per-scan RECIST features'),
    ('med_onc_note_level_dataset.csv',               '28,338 rows x 13 cols', 'per-note assessment outcomes'),
    ('pathology_report_level_dataset.csv',           '7,224 rows x 401 cols', 'per-report features'),
    ('tm_level_dataset.csv',                         '9,743 rows x 14 cols',  'CA15-3 / CA27-29 tumor markers'),
]
pd.DataFrame(files_used, columns=['filename','shape','description'])

,filename,shape,description
0,data_clinical_patient.txt,"1,134 rows x 42 cols",patient demographics + receptor status + workload
1,data_clinical_supp_survival.txt,"1,135 rows x 7 cols",OS + PFS_I_ADV + PFS_M_ADV
2,data_clinical_supp_survival_treatment.txt,"1,015 rows x 121 cols",per-regimen survival (skipped for brain-met an...
3,patient_level_dataset.csv,"1,130 rows x 50 cols",GENIE BPC patient frame
4,cancer_level_dataset_index.csv,"1,141 rows x 185 cols",GOLDMINE: per-organ time-to-met + stage_dx_iv
5,cancer_level_dataset_non_index.csv,191 rows x 112 cols,prior/concurrent non-breast cancers
6,data_clinical_sample.txt,"1,227 rows x 9 cols",sample metadata
7,cancer_panel_test_level_dataset.csv,"1,234 rows x 29 cols","one row per panel test, links sample to cancer"
8,data_gene_matrix.txt,"1,223 rows x 3 cols",which panel applied per sample
9,data_mutations_extended.txt,"7,051 rows x 64 cols","MAF with gnomAD AF, Polyphen, SIFT"


## 2. Per-component extraction

**Script:** `extract_genie_bpc_full.py` + `extract_genie_bpc_timelines.py`

Each source file was read with `comment='#'` and written to `data/processed/genie_bpc_v1_<name>.csv` for cBioPortal-style consistency. The CNA wide gene matrix was additionally **melted to long format** `(sampleId, hugoGeneSymbol, alteration)` filtering out neutral (`alteration == 0`) calls.

In [3]:
# List of per-component outputs (read-only check)
components = sorted(PROC.glob('genie_bpc_v1_*.csv'))
comp_df = pd.DataFrame([{'file': p.name, 'size_KB': round(p.stat().st_size/1024, 1)} for p in components])
comp_df

,file,size_KB
0,genie_bpc_v1_cancer_index.csv,1222.4
1,genie_bpc_v1_cancer_non_index.csv,100.1
2,genie_bpc_v1_clinical_master.csv,1923.6
3,genie_bpc_v1_clinical_patient.csv,369.7
4,genie_bpc_v1_clinical_sample.csv,116.8
5,genie_bpc_v1_clinical_supp_survival.csv,115.7
6,genie_bpc_v1_clinical_supp_survival_treatment.csv,372.3
7,genie_bpc_v1_cna.csv,1104.7
8,genie_bpc_v1_cna_gene_matrix.csv,2968.5
9,genie_bpc_v1_cna_seg.csv,2468.7


## 3. Clinical master join (cancer-level)

**Output:** `genie_bpc_v1_clinical_master.csv` (1,141 rows x 281 cols)

**Join logic** (one row per `(record_id, ca_seq)` index cancer):
```
cancer_index                                         # 1,141 rows x 185 cols (the goldmine)
  LEFT JOIN data_clinical_patient    ON record_id    # +41 patient cols (overlap suffixed __clinical_patient)
  LEFT JOIN data_clinical_supp_survival ON record_id # +6  cols   (suffixed __supp_survival)
  LEFT JOIN patient_level_dataset    ON record_id    # +49 cols   (suffixed __patient_bpc)
```

Overlapping non-key column names get suffixed with the source sheet name (e.g. `cohort__clinical_patient`) so nothing is silently overwritten.

## 4. Sample master join (sample-level)

**Output:** `genie_bpc_v1_sample_master.csv` (1,234 rows x 317 cols)

**Join logic** (one row per `SAMPLE_ID`):
```
data_clinical_sample                                          # 1,227 rows x 9 cols
  LEFT JOIN cancer_panel_test_level_dataset ON (SAMPLE_ID, record_id)  # +27 cols
  LEFT JOIN data_gene_matrix                ON SAMPLE_ID              # +2 cols (mutations/cna panel)
  LEFT JOIN clinical_master                 ON (record_id, ca_seq)     # +279 cols (the full clinical block)
```

**Key normalization:** `samp.PATIENT_ID` was renamed to `record_id` BEFORE the join with `cpt` (which already uses `record_id`). `cpt_genie_sample_id` was renamed to `SAMPLE_ID` to match the clinical sample frame.

In [4]:
# Inspect the sample_master shape and head identifiers
sm = pd.read_csv(PROC / 'genie_bpc_v1_sample_master.csv', nrows=3, low_memory=False)
print('shape (from one-row check):', sm.shape)
id_cols = [c for c in ('SAMPLE_ID', 'record_id', 'ca_seq', 'cpt_number', 'cohort', 'institution') if c in sm.columns]
sm[id_cols]

shape (from one-row check): (3, 317)


,SAMPLE_ID,record_id,ca_seq,cpt_number,cohort,institution
0,GENIE-MSK-P-0021826-T01-IM6,GENIE-MSK-P-0021826,0,1,BrCa,MSK
1,GENIE-MSK-P-0023994-T01-IM6,GENIE-MSK-P-0023994,0,1,BrCa,MSK
2,GENIE-MSK-P-0024083-T01-IM6,GENIE-MSK-P-0024083,2,1,BrCa,MSK


## 5. gnomeR gene-binary + Sanchez-Vega pathways

**Script:** `build_genie_bpc_sample_master_full.R` (R, uses gnomeR 1.2.2 installed in project renv)

**Output:** `genie_bpc_v1_sample_master_full.csv` (1,234 rows x 2,066 cols, 7.1 MB)

**gnomeR call:**
```r
bin <- create_gene_binary(
  samples             = unique(samp_master$SAMPLE_ID),  # 1,234 sample universe
  mutation            = mut,                            # MAF
  cna                 = cna,                            # long format
  fusion              = sv,                             # SV
  mut_type            = 'somatic_only',
  include_silent      = FALSE,                          # drops Silent/UTR/Intron/RNA/IGR
  snp_only            = FALSE,
  high_level_cna_only = TRUE,                           # only +2 (.Amp) and -2 (.Del)
  specify_panel       = 'no',                           # multi-panel cohort, no NA insertion
  recode_aliases      = 'no'
)
bin_paths <- add_pathways(bin, pathways = names(gnomeR::pathways))  # 10 Sanchez-Vega pathways
```

**Resulting feature counts:**
- 668 mutation gene cols (HUGO symbol, 0/1)
- 472 `.Amp` cols
- 251 `.Del` cols
- 343 `.fus` cols
- 10 `pathway_<X>` cols (`RTK/RAS`, `Nrf2`, `PI3K`, `TGFB`, `p53`, `Wnt`, `Myc`, `Cell cycle`, `Hippo`, `Notch`)

**Non-index cancer per-patient summary added:**
- `non_idx_n_cancers` - count of non-index cancers per patient
- `non_idx_any_heme` - any heme malignancy (TRUE/FALSE)
- `non_idx_any_brain` - any non-index cancer with brain involvement
- `non_idx_types` - semicolon-joined sorted unique `ca_type` values
- `had_prior_non_breast_cancer` - `non_idx_n_cancers > 0`

**Warning observed:** 22 samples had no alterations in any genomic file (mutations, CNA, or SV). gnomeR's `samples =` argument retains them as all-zero rows.

## 6. Final extraction + recoding

**Script:** `extract_variables_genie_data.py`

**Output:** `extracted_variables_genie_data.csv` (1,234 x 2,108)

Loads `genie_bpc_v1_sample_master_full.csv` + the full MAF, then applies:
- Per-sample MAF aggregation
- 20+ clinical recodings
- Brain-met cohort derivation
- Top-10 / top-5 gene pipeline
- Time-to-event column renames

### 6.1 Per-sample MAF aggregation

**Pre-filter:** drop rows where `Variant_Classification` in:
```
{Silent, 3'UTR, 5'UTR, 3'Flank, 5'Flank, Intron, RNA, IGR}
```
Also drop rows where `hugoGeneSymbol` or `sampleId` is NA.

Result: 6,840 mutations retained (from 7,051 source rows; 1,191 unique samples)

**Aggregation by `sampleId`:**

In [5]:
# Documentation only - logic that was applied (NOT re-run here)
AGGREGATION_LOGIC = '''
mut_summary = (
    maf_f.assign(t_alt_count=pd.to_numeric(maf_f.get('t_alt_count'), errors='coerce'))
         .groupby('sampleId')
         .agg(mutation_count_all_sites_sum=('hugoGeneSymbol', 'size'),
              t_alt_count_max=('t_alt_count', 'max'),
              genes=('hugoGeneSymbol', lambda s: ';'.join(sorted(s.dropna().unique()))))
         .reset_index()
         .rename(columns={'sampleId': 'SAMPLE_ID'})
)
# left-merge onto sample_master, fillna mutation_count_all_sites_sum with 0
'''
print(AGGREGATION_LOGIC)


mut_summary = (
    maf_f.assign(t_alt_count=pd.to_numeric(maf_f.get('t_alt_count'), errors='coerce'))
         .groupby('sampleId')
         .agg(mutation_count_all_sites_sum=('hugoGeneSymbol', 'size'),
              t_alt_count_max=('t_alt_count', 'max'),
              genes=('hugoGeneSymbol', lambda s: ';'.join(sorted(s.dropna().unique()))))
         .reset_index()
         .rename(columns={'sampleId': 'SAMPLE_ID'})
)
# left-merge onto sample_master, fillna mutation_count_all_sites_sum with 0



### 6.2 Variable recodings

Each recoding cell below documents source -> derivation -> output for the variables added by the extraction script. **These are reference cells; they do not re-run on the data.**

In [6]:
# 1. sample_type_bin (1 Primary, 2 Metastasis)
RECODE = '''
s = df['SAMPLE_TYPE_DETAILED'].astype(str).str.lower()
df['sample_type_bin'] = np.where(s.str.contains('primary'), 1,
                          np.where(s.str.contains('metast'), 2, np.nan))
'''
print(RECODE)


s = df['SAMPLE_TYPE_DETAILED'].astype(str).str.lower()
df['sample_type_bin'] = np.where(s.str.contains('primary'), 1,
                          np.where(s.str.contains('metast'), 2, np.nan))



In [7]:
# 2. age_cat (<50 / 50-70 / >70) from age_dx
RECODE = '''
df['age_dx_num'] = pd.to_numeric(df['age_dx'], errors='coerce')
df['age_cat'] = pd.cut(df['age_dx_num'],
                       bins=[-np.inf, 50, 70, np.inf],
                       labels=['<50', '50-70', '>70'], right=False)
# Ordered categorical: <50 < 50-70 < >70
'''
print(RECODE)


df['age_dx_num'] = pd.to_numeric(df['age_dx'], errors='coerce')
df['age_cat'] = pd.cut(df['age_dx_num'],
                       bins=[-np.inf, 50, 70, np.inf],
                       labels=['<50', '50-70', '>70'], right=False)
# Ordered categorical: <50 < 50-70 < >70



In [8]:
# 3. grade_ord (Low / Intermediate / High) from ca_grade
RECODE = '''
g = df['ca_grade'].astype(str)
# Regex match (case insensitive) on raw value
df['grade_ord'] = np.where(g.str.contains('Low', case=False, na=False) | g.str.match(r'^\\s*I\\s*$'),
                           'Low',
                  np.where(g.str.contains('Intermediate', case=False, na=False) | g.str.contains(r'^\\s*II\\s*$', na=False),
                           'Intermediate',
                    np.where(g.str.contains('High', case=False, na=False) | g.str.contains(r'^\\s*III\\s*$', na=False),
                             'High', None)))
df['grade_ord'] = pd.Categorical(df['grade_ord'], categories=['Low', 'Intermediate', 'High'], ordered=True)
'''
print(RECODE)


g = df['ca_grade'].astype(str)
# Regex match (case insensitive) on raw value
df['grade_ord'] = np.where(g.str.contains('Low', case=False, na=False) | g.str.match(r'^\s*I\s*$'),
                           'Low',
                  np.where(g.str.contains('Intermediate', case=False, na=False) | g.str.contains(r'^\s*II\s*$', na=False),
                           'Intermediate',
                    np.where(g.str.contains('High', case=False, na=False) | g.str.contains(r'^\s*III\s*$', na=False),
                             'High', None)))
df['grade_ord'] = pd.Categorical(df['grade_ord'], categories=['Low', 'Intermediate', 'High'], ordered=True)



In [9]:
# 4. stage_diag_group from stage_dx (uppercase + strip first)
RECODE = '''
s = df['stage_dx'].astype(str).str.upper().str.strip()
df['stage_diag_group'] = np.where(s.isin(['I', 'IA', 'IB', 'STAGE I']), 'Stage I',
                           np.where(s.isin(['II', 'IIA', 'IIB', 'STAGE II']), 'Stage II',
                             np.where(s.isin(['III', 'IIIA', 'IIIB', 'IIIC', 'STAGE III']), 'Stage III',
                               np.where(s.isin(['IV', 'STAGE IV']), 'Stage IV', None))))
df['stage_diag_group'] = pd.Categorical(df['stage_diag_group'],
                                        categories=['Stage I', 'Stage II', 'Stage III', 'Stage IV'],
                                        ordered=True)
'''
print(RECODE)


s = df['stage_dx'].astype(str).str.upper().str.strip()
df['stage_diag_group'] = np.where(s.isin(['I', 'IA', 'IB', 'STAGE I']), 'Stage I',
                           np.where(s.isin(['II', 'IIA', 'IIB', 'STAGE II']), 'Stage II',
                             np.where(s.isin(['III', 'IIIA', 'IIIB', 'IIIC', 'STAGE III']), 'Stage III',
                               np.where(s.isin(['IV', 'STAGE IV']), 'Stage IV', None))))
df['stage_diag_group'] = pd.Categorical(df['stage_diag_group'],
                                        categories=['Stage I', 'Stage II', 'Stage III', 'Stage IV'],
                                        ordered=True)



In [10]:
# 5. stage_iv_bin from stage_dx_iv (binary)
RECODE = '''
df['stage_iv_bin'] = (df['stage_dx_iv'].astype(str).str.strip().str.lower()
                         .isin(['stage iv', 'iv', 'yes', 'true', '1'])
                         .astype(int))
'''
print(RECODE)


df['stage_iv_bin'] = (df['stage_dx_iv'].astype(str).str.strip().str.lower()
                         .isin(['stage iv', 'iv', 'yes', 'true', '1'])
                         .astype(int))



In [11]:
# 6. receptor_primary_cat (4-class HR/HER2) from bca_subtype
RECODE = '''
sub = df['bca_subtype'].astype(str)
df['receptor_primary_cat'] = np.where(
    sub.isin(['HR+, HER2-', 'HR+/HER2-']), 'HR+/HER2-',
    np.where(sub.isin(['HR+, HER2+', 'HR+/HER2+']), 'HR+/HER2+',
      np.where(sub.isin(['HR-, HER2+', 'HR-/HER2+']), 'HR-/HER2+',
        np.where(sub.isin(['TNBC', 'Triple Negative', 'HR-, HER2-', 'HR-/HER2-']),
                 'Triple Negative', None))))
df['receptor_primary_cat'] = pd.Categorical(
    df['receptor_primary_cat'],
    categories=['HR+/HER2-', 'HR+/HER2+', 'HR-/HER2+', 'Triple Negative']
)
'''
print(RECODE)


sub = df['bca_subtype'].astype(str)
df['receptor_primary_cat'] = np.where(
    sub.isin(['HR+, HER2-', 'HR+/HER2-']), 'HR+/HER2-',
    np.where(sub.isin(['HR+, HER2+', 'HR+/HER2+']), 'HR+/HER2+',
      np.where(sub.isin(['HR-, HER2+', 'HR-/HER2+']), 'HR-/HER2+',
        np.where(sub.isin(['TNBC', 'Triple Negative', 'HR-, HER2-', 'HR-/HER2-']),
                 'Triple Negative', None))))
df['receptor_primary_cat'] = pd.Categorical(
    df['receptor_primary_cat'],
    categories=['HR+/HER2-', 'HR+/HER2+', 'HR-/HER2+', 'Triple Negative']
)



In [12]:
# 7. race_clean from PRIMARY_RACE (regex collapse)
RECODE = '''
r = df['PRIMARY_RACE'].astype(str)
df['race_clean'] = np.where(r.str.contains('White|Middle East', case=False, na=False), 'White',
                      np.where(r.str.contains('Black|African', case=False, na=False), 'Black',
                        np.where(r.str.contains('Asian|Indian|Chinese', case=False, na=False), 'Asian',
                          np.where(r.str.contains('Native|Alaska', case=False, na=False), 'Native American',
                                   None))))
'''
print(RECODE)


r = df['PRIMARY_RACE'].astype(str)
df['race_clean'] = np.where(r.str.contains('White|Middle East', case=False, na=False), 'White',
                      np.where(r.str.contains('Black|African', case=False, na=False), 'Black',
                        np.where(r.str.contains('Asian|Indian|Chinese', case=False, na=False), 'Asian',
                          np.where(r.str.contains('Native|Alaska', case=False, na=False), 'Native American',
                                   None))))



In [13]:
# 8. ethnicity_clean from ETHNICITY
RECODE = '''
e = df['ETHNICITY'].astype(str)
df['ethnicity_clean'] = np.where(
    e.str.contains('Non-Spanish|Non-Hispanic', case=False, na=False), 'Non-Hispanic',
    np.where(e.str.contains('Hispanic|Latino|Cuban|Mexican|Puerto', case=False, na=False),
             'Hispanic', None))
'''
print(RECODE)


e = df['ETHNICITY'].astype(str)
df['ethnicity_clean'] = np.where(
    e.str.contains('Non-Spanish|Non-Hispanic', case=False, na=False), 'Non-Hispanic',
    np.where(e.str.contains('Hispanic|Latino|Cuban|Mexican|Puerto', case=False, na=False),
             'Hispanic', None))



In [14]:
# 9. her2_status_bin from ca_bca_her_summ
RECODE = '''
h = df['ca_bca_her_summ'].astype(str)
df['her2_status_bin'] = np.where(
    h.str.contains('positive|amplif|equivocal_pos', case=False, na=False), 1,
    np.where(h.str.contains('negative|not amplif', case=False, na=False), 0, np.float64('nan')))
'''
print(RECODE)


h = df['ca_bca_her_summ'].astype(str)
df['her2_status_bin'] = np.where(
    h.str.contains('positive|amplif|equivocal_pos', case=False, na=False), 1,
    np.where(h.str.contains('negative|not amplif', case=False, na=False), 0, np.float64('nan')))



In [15]:
# 10. OS status recoding (factor + binary) from os_dx_status
RECODE = '''
s = df['os_dx_status'].astype(str)
df['os_status_f'] = np.where(s.str.startswith('0'), 'Alive',
                      np.where(s.str.startswith('1'), 'Deceased', None))
df['os_status_bin'] = pd.to_numeric(df['os_dx_status'].astype(str).str[0], errors='coerce')
'''
print(RECODE)


s = df['os_dx_status'].astype(str)
df['os_status_f'] = np.where(s.str.startswith('0'), 'Alive',
                      np.where(s.str.startswith('1'), 'Deceased', None))
df['os_status_bin'] = pd.to_numeric(df['os_dx_status'].astype(str).str[0], errors='coerce')



In [16]:
# 11. PFS event indicators (binary 0/1) from cBioPortal status strings
RECODE = '''
df['pfs_i_event_bin'] = pd.to_numeric(df['pfs_i_adv_status'].astype(str).str[0], errors='coerce')
df['pfs_m_event_bin'] = pd.to_numeric(df['pfs_m_adv_status'].astype(str).str[0], errors='coerce')
'''
print(RECODE)


df['pfs_i_event_bin'] = pd.to_numeric(df['pfs_i_adv_status'].astype(str).str[0], errors='coerce')
df['pfs_m_event_bin'] = pd.to_numeric(df['pfs_m_adv_status'].astype(str).str[0], errors='coerce')



In [17]:
# 12. mutation_count_q and t_alt_count_q (quartiles)
RECODE = '''
df['mutation_count_q'] = pd.qcut(df['mutation_count_all_sites_sum'], q=4,
                                 labels=['Q1', 'Q2', 'Q3', 'Q4'], duplicates='drop')
df['t_alt_count_q']    = pd.qcut(df['t_alt_count_max'], q=4,
                                 labels=['Q1', 'Q2', 'Q3', 'Q4'], duplicates='drop')
'''
print(RECODE)


df['mutation_count_q'] = pd.qcut(df['mutation_count_all_sites_sum'], q=4,
                                 labels=['Q1', 'Q2', 'Q3', 'Q4'], duplicates='drop')
df['t_alt_count_q']    = pd.qcut(df['t_alt_count_max'], q=4,
                                 labels=['Q1', 'Q2', 'Q3', 'Q4'], duplicates='drop')



### 6.3 Brain-met cohort derivation

The analytic cohort split. `dist_mets_brain_cns` is the cancer-level cumulative flag from `cancer_level_dataset_index` (covers any time post-dx). `DMETS_DX_BRAIN` from `data_clinical_patient` is at-dx only. The union captures both.

In [18]:
RECODE = '''
brain_overall = df['dist_mets_brain_cns'].fillna(0).astype(int) == 1
brain_at_dx   = (df.get('DMETS_DX_BRAIN', pd.Series(['']*len(df)))
                   .astype(str).str.lower() == 'yes')
df['any_brain_met']   = (brain_overall | brain_at_dx).astype(int)
df['brain_met_at_dx'] = brain_at_dx.astype(int)

# met_loc: Brain / Other / None (priority: Brain > Other > None)
other_organ_cols = [c for c in df.columns if c.startswith('dist_mets_')
                    and c not in {'dist_mets_brain_cns'}
                    and not c.startswith('dx_to_dist_mets_')]
has_other_met = df[other_organ_cols].fillna(0).astype(int).max(axis=1) == 1
df['met_loc'] = np.where(df['any_brain_met'] == 1, 'Brain',
                  np.where(has_other_met, 'Other', 'None'))
df['met_loc'] = pd.Categorical(df['met_loc'], categories=['Brain', 'Other', 'None'])
'''
print(RECODE)


brain_overall = df['dist_mets_brain_cns'].fillna(0).astype(int) == 1
brain_at_dx   = (df.get('DMETS_DX_BRAIN', pd.Series(['']*len(df)))
                   .astype(str).str.lower() == 'yes')
df['any_brain_met']   = (brain_overall | brain_at_dx).astype(int)
df['brain_met_at_dx'] = brain_at_dx.astype(int)

# met_loc: Brain / Other / None (priority: Brain > Other > None)
other_organ_cols = [c for c in df.columns if c.startswith('dist_mets_')
                    and c not in {'dist_mets_brain_cns'}
                    and not c.startswith('dx_to_dist_mets_')]
has_other_met = df[other_organ_cols].fillna(0).astype(int).max(axis=1) == 1
df['met_loc'] = np.where(df['any_brain_met'] == 1, 'Brain',
                  np.where(has_other_met, 'Other', 'None'))
df['met_loc'] = pd.Categorical(df['met_loc'], categories=['Brain', 'Other', 'None'])



In [19]:
# Confirm cohort distribution from the final output file (read-only)
df = pd.read_csv(PROC / 'extracted_variables_genie_data.csv',
                 usecols=['SAMPLE_ID', 'record_id', 'any_brain_met', 'brain_met_at_dx', 'met_loc'],
                 low_memory=False)
print('Cohort split (samples):')
print(df['met_loc'].value_counts(dropna=False))
print()
print('any_brain_met == 1 patients:', df.loc[df['any_brain_met']==1, 'record_id'].nunique())
print('brain_met_at_dx == 1 patients:', df.loc[df['brain_met_at_dx']==1, 'record_id'].nunique())

Cohort split (samples):
met_loc
Other    571
Brain    393
NaN      270
Name: count, dtype: int64

any_brain_met == 1 patients: 362
brain_met_at_dx == 1 patients: 9


### 6.4 Top-gene pipeline

Step 1: identify genuine mutation gene columns by intersecting `df.columns` with the MAF gene set (`hugoGeneSymbol`) AND requiring integer dtype. This avoids matching uppercase clinical columns like `ETHNICITY`, `SEX`, `CENTER`.

Step 2: in the brain-met sub-cohort, count `n_brain_met_samples_mutated` per gene; sort descending.

Step 3: take top 10 -> `TOP10_GENES`; first 5 -> `TOP5_GENES`. Top-10 derived on GENIE BPC:

```
TP53, PIK3CA, GATA3, CDH1, PTEN, NF1, ARID1A, KMT2D, BRCA2, KMT2C
```

Step 4: add per-sample binary indicators.

In [20]:
RECODE = '''
# Identify gene cols (intersect with MAF gene list + int dtype)
maf_gene_set = set(maf['hugoGeneSymbol'].dropna().astype(str).unique())
mut_gene_cols = [c for c in df.columns
                 if c in maf_gene_set
                 and pd.api.types.is_integer_dtype(df[c])
                 and not c.endswith(('.Amp', '.Del', '.fus'))]

# Restrict to brain-met cohort, count gene prevalence
brain_df = df[df['any_brain_met'] == 1]
gene_prev = (brain_df[mut_gene_cols].sum()
                      .sort_values(ascending=False)
                      .to_frame('n_brain_met_samples_mutated'))
gene_prev['pct_brain_met'] = gene_prev['n_brain_met_samples_mutated'] / len(brain_df)
gene_prev['n_total_samples_mutated'] = df[gene_prev.index.tolist()].sum().values
gene_prev['pct_total'] = gene_prev['n_total_samples_mutated'] / len(df)

top10_genes = gene_prev.head(10).index.tolist()
top5_genes  = top10_genes[:5]

# Per-sample binary indicators
for g in top10_genes:
    df[f'G_top10_{g}'] = df[g].astype(int)
df['top5_any_mutated']  = (df[top5_genes].sum(axis=1)  > 0).astype(int)
df['top10_any_mutated'] = (df[top10_genes].sum(axis=1) > 0).astype(int)
df['top5_n_mutated']    =  df[top5_genes].sum(axis=1).astype(int)
df['top10_n_mutated']   =  df[top10_genes].sum(axis=1).astype(int)
'''
print(RECODE)


# Identify gene cols (intersect with MAF gene list + int dtype)
maf_gene_set = set(maf['hugoGeneSymbol'].dropna().astype(str).unique())
mut_gene_cols = [c for c in df.columns
                 if c in maf_gene_set
                 and pd.api.types.is_integer_dtype(df[c])
                 and not c.endswith(('.Amp', '.Del', '.fus'))]

# Restrict to brain-met cohort, count gene prevalence
brain_df = df[df['any_brain_met'] == 1]
gene_prev = (brain_df[mut_gene_cols].sum()
                      .sort_values(ascending=False)
                      .to_frame('n_brain_met_samples_mutated'))
gene_prev['pct_brain_met'] = gene_prev['n_brain_met_samples_mutated'] / len(brain_df)
gene_prev['n_total_samples_mutated'] = df[gene_prev.index.tolist()].sum().values
gene_prev['pct_total'] = gene_prev['n_total_samples_mutated'] / len(df)

top10_genes = gene_prev.head(10).index.tolist()
top5_genes  = top10_genes[:5]

# Per-sample binary indicators
for g in top10_genes:
    df[f'G_top10_{g}'] = df[g].astype(i

In [21]:
# Show the actual top-10 gene prevalence as derived
gene_prev = pd.read_csv(PROC / 'extracted_variables_genie_gene_prev_brain_met.csv')
gene_prev.head(10)

,gene,n_brain_met_samples_mutated,pct_brain_met,n_total_samples_mutated,pct_total
0,TP53,206,0.524173,570,0.461912
1,PIK3CA,112,0.284987,353,0.286062
2,GATA3,46,0.117048,174,0.141005
3,CDH1,35,0.089059,122,0.098865
4,PTEN,26,0.066158,73,0.059157
5,NF1,25,0.063613,63,0.051053
6,ARID1A,24,0.061069,84,0.068071
7,KMT2D,24,0.061069,78,0.063209
8,BRCA2,24,0.061069,66,0.053485
9,KMT2C,23,0.058524,61,0.049433


### 6.5 Time-to-event variables (modeling-ready)

Brain-met time-to-event uses overall follow-up time (`tt_os_dx_mos`) as the censoring time for patients without a brain met.

In [22]:
RECODE = '''
df['tt_brain_met_mos'] = np.where(
    df['any_brain_met'] == 1,
    pd.to_numeric(df.get('dx_to_dist_mets_brain_cns_mos'), errors='coerce'),  # time to brain met
    pd.to_numeric(df.get('tt_os_dx_mos'), errors='coerce')                   # censoring time
)
df['brain_met_event'] = df['any_brain_met']

# Analyst-friendly renames
rename_for_analysis = {
    'tt_os_dx_mos':                'OS_months',
    'tt_pfs_i_adv_mos':            'PFS_imaging_months',
    'tt_pfs_m_adv_mos':            'PFS_medonc_months',
    'dx_to_dist_mets_brain_cns_mos': 'time_to_brain_met_mos',
}
for old, new in rename_for_analysis.items():
    if old in df.columns and new not in df.columns:
        df[new] = df[old]
'''
print(RECODE)


df['tt_brain_met_mos'] = np.where(
    df['any_brain_met'] == 1,
    pd.to_numeric(df.get('dx_to_dist_mets_brain_cns_mos'), errors='coerce'),  # time to brain met
    pd.to_numeric(df.get('tt_os_dx_mos'), errors='coerce')                   # censoring time
)
df['brain_met_event'] = df['any_brain_met']

# Analyst-friendly renames
rename_for_analysis = {
    'tt_os_dx_mos':                'OS_months',
    'tt_pfs_i_adv_mos':            'PFS_imaging_months',
    'tt_pfs_m_adv_mos':            'PFS_medonc_months',
    'dx_to_dist_mets_brain_cns_mos': 'time_to_brain_met_mos',
}
for old, new in rename_for_analysis.items():
    if old in df.columns and new not in df.columns:
        df[new] = df[old]



## 7. Final output summary

**`extracted_variables_genie_data.csv`**: 1,234 samples x 2,108 cols

Variable groups (approximate):
- Clinical & cancer-level (228 cols)
- Survival endpoints (~50 cols across OS / PFS_imaging / PFS_medonc / OS_adv / PFS_or_and / brain_met_TTE / per-organ TTE)
- Genomic gene-binary (1,734 cols: 668 mut + 472 .Amp + 251 .Del + 343 .fus)
- Pathways (10)
- Top-gene indicators (10 G_top10_*, plus top5/top10 summaries)
- MAF-derived (`mutation_count_all_sites_sum`, `t_alt_count_max`, `genes`, plus quartiles)
- Recoded clinical (sample_type_bin, age_cat, grade_ord, stage_diag_group, stage_iv_bin, receptor_primary_cat, race_clean, ethnicity_clean, her2_status_bin, os_status_*, pfs_*_event_bin)
- Brain-met cohort (`any_brain_met`, `brain_met_at_dx`, `met_loc`, `tt_brain_met_mos`, `brain_met_event`)
- Non-index cancer summary (5)

**Companion files in `data/processed/`:**
- `extracted_variables_genie_top_genes.txt` - one gene per line, the top-10 list
- `extracted_variables_genie_gene_prev_brain_met.csv` - full gene prevalence ranking with cols `gene, n_brain_met_samples_mutated, pct_brain_met, n_total_samples_mutated, pct_total`
- `extracted_variables_genie_dictionary.csv` - per-variable recoding audit trail (26 rows: variable, original, mapped, note)

In [23]:
# Read-only confirmation that the final file is on disk
final = PROC / 'extracted_variables_genie_data.csv'
header_only = pd.read_csv(final, nrows=0, low_memory=False)
print(f'File:        {final}')
print(f'Size:        {final.stat().st_size/1e6:.1f} MB')
print(f'Columns:     {header_only.shape[1]}')
print()
print('Companion files:')
for p in sorted(PROC.glob('extracted_variables_genie*')):
    print(f'  {p.name:55s} {p.stat().st_size/1024:>8.1f} KB')

File:        /Users/robertjames/Documents/Documents - Robert’s iMac/Research Projects/MSKCC Research Fellowship/Projects/genie_tcga_impact_brain_mets/data/processed/extracted_variables_genie_data.csv
Size:        7.2 MB
Columns:     2108

Companion files:
  extracted_variables_genie_data.csv                        7031.1 KB
  extracted_variables_genie_dictionary.csv                     7.7 KB
  extracted_variables_genie_gene_prev_brain_met.csv           31.7 KB
  extracted_variables_genie_top_genes.txt                      0.1 KB


## 8. Cohort validation tables

These are the QC summaries printed at the end of the extraction script. All read-only here.

In [24]:
df = pd.read_csv(PROC / 'extracted_variables_genie_data.csv',
                 usecols=['any_brain_met', 'top5_any_mutated', 'receptor_primary_cat', 'met_loc', 'record_id'],
                 low_memory=False)

print('any_brain_met x top5_any_mutated (samples):')
print(pd.crosstab(df['any_brain_met'], df['top5_any_mutated'], margins=True, margins_name='total'))
print()
print('met_loc distribution (samples):')
print(df['met_loc'].value_counts(dropna=False))
print()
print('receptor_primary_cat x any_brain_met (samples):')
print(pd.crosstab(df['receptor_primary_cat'], df['any_brain_met']))

any_brain_met x top5_any_mutated (samples):
top5_any_mutated    0    1  total
any_brain_met                    
0                 175  666    841
1                  76  317    393
total             251  983   1234

met_loc distribution (samples):
met_loc
Other    571
Brain    393
NaN      270
Name: count, dtype: int64

receptor_primary_cat x any_brain_met (samples):
any_brain_met           0    1
receptor_primary_cat          
HR+/HER2+              90   54
HR+/HER2-             539  211
HR-/HER2+              42   40
Triple Negative       116   70


## 9. Caveats / known issues

1. **`PATIENT_ID` is renamed to `record_id`** in the master joins (canonical GENIE BPC key). Original `PATIENT_ID` is not retained in the final frame. Use `record_id` as the patient key.

2. **Multi-panel coverage:** gnomeR was called with `specify_panel = 'no'` because the cohort spans MSK-IMPACT (341/410/468), DFCI-ONCOPANEL (1/2/3/3.1), and VICC (D2/T5A/T7) panels with different gene coverage. A gene shows `0` whether the sample was genuinely wild-type OR the gene was not on that panel. To get true NA for non-paneled genes, construct a custom `(sample_id, gene_panel)` mapping from `genie_bpc_v1_gene_matrix.csv` + the 10 panel files in the source release.

3. **gnomAD AF / Polyphen filtering not applied.** The MAF columns are retained in `genie_bpc_v1_mutations.csv` if stricter germline filtering is needed before re-running.

4. **22 samples have zero alterations** across MAF, CNA, and SV. Kept as all-zero rows per gnomeR's `samples =` argument.

5. **Top-10 gene list is cohort-specific.** Derived on the GENIE BPC brain-met sub-cohort; do not copy these specific genes when applying the pipeline to a different cohort. Re-derive from the new cohort's MAF.

6. **The 'Other_clinical: 720' category** in the classification audit is mostly mis-categorized gene-binary mutation cols (bare HUGO symbol names match an 'uppercase, no underscore' regex). True genomic count: 1,734 gene-binary + 10 pathway + 10 top-gene indicators + 9 MAF summary.

## 10. Cross-reference to harmonization spec

If applying this pipeline to a **new cohort**, follow `src/data collection and processing/harmonization_spec.md` which contains:
- Required input columns by section
- All categorical recoding mappings as lookup tables
- gnomeR call settings
- Required output file structure

The harmonization spec is the LLM-feedable version of this notebook.

## 11. Appendix: R lubridate date handling reference

The GENIE BPC frame uses pre-computed time-to-event columns in months (`tt_brain_met_mos`, `OS_months`, etc.) rather than raw dates, so the date-parsing logic below is not part of the build pipeline. It is preserved as a working reference for any downstream R script that needs to parse date strings, build dates from components, extract components, or compute time spans.

```r
############################################################
# DATE HANDLING TEMPLATE - Working With Dates and Times
# Section 3.6 - Lubridate & nycflights13 Example
############################################################

# 1. Setup
# install.packages('lubridate')
# install.packages('nycflights13')
library(lubridate)
library(dplyr)
library(nycflights13)

# 2. Creating Date and Date-Time Objects
# --- From strings ---
date_ymd <- ymd("1988-09-29")           # Year-Month-Day
date_mdy <- mdy("September 29th, 1988") # Month-Day-Year
date_dmy <- dmy("29-Sep-1988")          # Day-Month-Year
date_ymd; date_mdy; date_dmy

# --- From strings including time ---
datetime_ymd_hms <- ymd_hms("1988-09-29 20:11:59")
datetime_ymd_hms

# 3. Creating Dates/Date-Times from Individual Parts
# --- Using make_date() ---
flights_dates <- flights %>%
  select(year, month, day) %>%
  mutate(departure = make_date(year, month, day))
head(flights_dates)

# --- Using make_datetime() ---
flights_datetimes <- flights %>%
  select(year, month, day, hour, minute) %>%
  mutate(departure = make_datetime(year, month, day, hour, minute))
head(flights_datetimes)

# 4. Extracting Components from Dates
mydate <- ymd("1988-09-29")
year_info     <- year(mydate)              # Extract year
day_info      <- mday(mydate)              # Extract day of the month
weekday_num   <- wday(mydate)              # Extract weekday (numeric)
weekday_label <- wday(mydate, label = TRUE)# Extract weekday (labeled)
year_info; day_info; weekday_num; weekday_label

# 5. Working with Time Spans
birthday      <- ymd("1988-09-29")
age_diff      <- today() - birthday         # in days
age_duration  <- as.duration(age_diff)      # in seconds
years_old     <- as.numeric(age_duration) / (60 * 60 * 24 * 365)
age_diff; age_duration; years_old

# 6. Additional Examples
today_plus_30      <- today() + days(30)
today_minus_1year  <- today() - years(1)
today_plus_30; today_minus_1year

############################################################
# END OF TEMPLATE
############################################################
```

**Usage notes for this project:**
- The GENIE BPC source files store time relative to dx in `*_days`, `*_mos`, `*_yrs` triplets, so absolute-date parsing is rarely needed. Use the pre-computed `_mos` columns for survival modeling.
- If you DO need to parse `CPT_SEQ_DATE` or `cpt_seq_date` (the only absolute-date columns in the frame), `lubridate::ymd()` handles the source ISO format directly.
- For age-at-dx, `age_dx` is already a numeric in the frame; no date arithmetic needed.